# Setup

In [1]:
from pprint import pprint
import os, math
import pandas as pd
import torch
from transformers.utils import logging
from transformers import set_seed

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = device.type == 'cuda' and torch.cuda.is_bf16_supported()

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# Seed.
seed = 42
set_seed(seed)

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0916 09:12:13.271000 7900 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# Basics

## Steps

1. Parse and cleaning
2. Chunking

## Chunking

- Fixed length
- Structure-aware chunking
- Semantic chunking

## Prompt Engineering

- Query
  - Query rewriting: one better query.
  - Query expansion: enrich one query with additional terms.
  - Multi-query retrieval: create multiple alternative queries and combine.
  - Query decomposition: split complex query into simpler subqueries.
    - Multi-hop / iterative RAG
- HyDE: Hypothetical Document Embeddings
  - Instead of directly embedding the user query,
  - ask an LLM to generate a hypothetical answer/document,
  - and use this embedding for search.

## Retrieval

- Dense retrieval
  - Convert document/query into embeddings.
  - Retrieve the documents using similarity.
- Sparse retrieval (BM25)
  - Best Matching 25.
  - Search in the document, for the keywords in the query.
- Hybrid retrieval
  - Reciprocal Rank Fusion, $RRF(d) = \displaystyle \sum_r {\frac{1}{k + rank_r(d)}}$
- Metadata filtering
  - Filter by the given structured metadata.
- Top-$K$
  - Retrieve top-$K$ similar documents.
- Reranking
  - Bi-encoder
    - Embed query/document separately.
    - Search the document based on similarity and retrieve.
    - Recall: did we find related candidates in document?
  - Cross-encoder
    - Calculate relevance score of the query and retrieved candidates.
    - Query and candidates interact directly through attention.
    - Precision: are the candidates actually useful?

## Context Construction

- Ordering by score
- Deduplication
- Truncation according to the context budget
- Rearrange important evidences
  - Lost-in-the-middle problem: LLMs often treat the information near the beginning/end of a long context.
- Context compression
- Grounded generation
  - The answer should be supported by the retrieval, rather a pure LLM's internal knowledge.
- Abstention
  - If there is not enough evidence, explicitly says "I don't know" rather hallucination.
- Parent-child retrieval
  - Retrieve a child chunk -> add the whole parent chunk.

## RAG Evaluation

- Retriever
  - Recall@k
    - How much are relevant evidences in the top-k?
    - Total 5 relevences.
    - Top-3 = [A, e1, e2] -> 2/5
  - Hit rate@k
    - Is there at least one relevant evidence?
    - Top-3 = [A, e1, B] -> True = 1
  - Precision@k
    - What fraction of the top-k are relevant?
    - top-3 = [A, relevant, B] -> 1/3
  - MRR: Mean Reciprocal Rank
    - Rank of the first relevant evidence.
    - top-5 = [A, B, C, relevant, D] -> 4/5
  - nDCG: normalized Discounted Cumulative Gain
    - Evaluate the entire retrieved list -> more relevant should get higher rank.
  - Context relevance
    - Was the retrieved evidence actually useful?
- Generator
  - Answer quality
  - Faithfulness / groundness: is the answer supported by retrieved evidence?
  - Context relevance

## Advanced RAG

- Self-RAG
  - Self-Reflective RAG
  - Instead of always retrieving, the model learns to decide.
- CRAG
  - Corrective RAG
  - What should the LLM do when retrieved documents are poor?
- Graph RAG
  - Graph-based RAG
  - Stores an entity and relationship too.
  - Retrieve connected entities / relationships / supporting texts too.
- Agentic RAG
  - An agent actively controls retrieval.

## RAG Framework

- Qdrant: for general RAG system
  - `pip install qdrant-client`
- faiss: low-level api for vector search

# Example - Simple RAG

- Dataset: SciFact
  - 5,183 corpus documents
  - 1,109 queries
  - 339 relevances (300 unique test queries)
- Retrieval: dense retrieval

## Dataset

In [2]:
from datasets import load_dataset

# Corpus.
corpus = load_dataset(
    'BeIR/scifact',
    'corpus',
    split='corpus',
)

# Queries.
queries = load_dataset(
    'BeIR/scifact',
    'queries',
    split='queries',
)

# Relevances. score = 1 -> relevant.
qrels = load_dataset(
    'BeIR/scifact-qrels',
    split='test',
)

# Example.
rel = qrels[0]
query_id = str(rel['query-id'])
corpus_id = str(rel['corpus-id'])

query = next(
    row for row in queries
    if str(row['_id']) == query_id
)
document = next(
    row for row in corpus
    if str(row['_id']) == corpus_id
)

print('Query:')
print(query['text'])

print('\nRelevant document:')
pprint(document['title'])
pprint(document['text'])

Query:
0-dimensional biomaterials show inductive properties.

Relevant document:
('New opportunities: the use of nanotechnologies to manipulate and track stem '
 'cells.')
('Nanotechnologies are emerging platforms that could be useful in measuring, '
 'understanding, and manipulating stem cells. Examples include magnetic '
 'nanoparticles and quantum dots for stem cell labeling and in vivo tracking; '
 'nanoparticles, carbon nanotubes, and polyplexes for the intracellular '
 'delivery of genes/oligonucleotides and protein/peptides; and engineered '
 'nanometer-scale scaffolds for stem cell differentiation and transplantation. '
 'This review examines the use of nanotechnologies for stem cell tracking, '
 'differentiation, and transplantation. We further discuss their utility and '
 'the potential concerns regarding their cytotoxicity.')


## Embedding

- Sentence transformer
  - An embedding model designed to turn an entire sentence/document into one fixed-size embedding vector.
- `BAAI/bge-small-en-v1.5`
  - A small RAG embedding model for learning.
> Note) If an embedding is L2-normalized, i.e. $e' = \frac{e}{||e||_2}$, then $||e'||_2 = 1$ and $cos(e_q, e_d) = q^T d$

In [ ]:
from sentence_transformers import SentenceTransformer

# Embedding model.
embedding_model = SentenceTransformer(
    'BAAI/bge-small-en-v1.5',
).to(device)

# Build one text string per document.
doc_texts = []

for row in corpus:
    title = row['title']
    text = row['text']

    doc_text = title + '\n' + text
    doc_texts.append(doc_text)

# Encode.
doc_embeddings = embedding_model.encode(
    doc_texts,
    batch_size=128,
    normalize_embeddings=True,      # L2-normalization.
    convert_to_numpy=True,
    show_progress_bar=True,
)

print(doc_embeddings.shape)     # (N, d) = (5_183, 384)

Batches: 100%|██████████| 41/41 [00:05<00:00,  7.63it/s]

(5183, 384)


## Client

```python
client = QdrantClient(
    path=...,           # persist in disk.
    path=':memory:',    # in-memory -> deleted after the process ends.
    url=...,            # qdrant server.
)
```

In [ ]:
from qdrant_client import QdrantClient

# Client.
collection_name = 'scifact_dense'
client = QdrantClient(
    path='../tmp/outputs/qdrant_scifact',   # persist in disk.
)

## Collection

In [ ]:
from qdrant_client import models

# Delete the collection if already exists.
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

# Create a collection.
vector_size = doc_embeddings.shape[1]       # (N, d)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE,    # cosine similarity.
    )
)

True

## Insert

In [ ]:
# One point.
row = corpus[0]
embedding = doc_embeddings[0]

point = models.PointStruct(
    id=int(row['_id']),
    vector=embedding.tolist(),
    payload={   # metadata.
        'title': row['title'],
        'text': row['text'],
    }
)

client.upsert(
    collection_name=collection_name,
    points=[point]
)

# Entire corpus.
points = []

for row, embedding in zip(corpus, doc_embeddings):
    point = models.PointStruct(
        id=int(row['_id']),
        vector=embedding.tolist(),
        payload={
            'title': row['title'],
            'text': row['text'],
        },
    )
    points.append(point)

client.upload_points(
    collection_name=collection_name,
    points=points,
    batch_size=128,
)

# Count.
count = client.count(
    collection_name=collection_name,
    exact=True,     # if False, use approximation.
)

print(f"Count: {count.count}")

# Close the client.
# client.close()

Count: 5183


## Query Embedding

In [7]:
query = queries[0]['text']

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

## Retrieval

In [15]:
results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=3,                # top-3 documents.
    with_payload=True,      # returns the stored title/text too.
)
retrievals = results.points

for rank, result in enumerate(retrievals, start=1):
    print(f'Rank: {rank}')
    print(f'Doc ID: {result.id}')
    print(f'Score: {result.score:.4f}')
    print(f"Title: {result.payload['title']}")
    print(f"Text: {result.payload['text'][:50]} ...")
    print()

Rank: 1
Doc ID: 17388232
Score: 0.7618
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates
Text: We report the establishment of a library of microm ...

Rank: 2
Doc ID: 4346436
Score: 0.7549
Title: Nonlinear Elasticity in Biological Gels
Text: Unlike most synthetic materials, biological materi ...

Rank: 3
Doc ID: 29638116
Score: 0.7102
Title: Complex Tissue and Disease Modeling using hiPSCs.
Text: Defined genetic models based on human pluripotent  ...



## Evaluation

In [ ]:
import ir_measures
from ir_measures import R, RR, nDCG, Success

# 1. Test queries.
test_query_ids = sorted({
    str(row['query-id'])
    for row in qrels
})
query_lookup = {
    str(row['_id']): row['text']
    for row in queries
}
test_query_texts = [
    query_lookup[query_id]
    for query_id in test_query_ids
]
print('Test queries:', len(test_query_ids))

# 2. Embedding.
query_embeddings = embedding_model.encode(
    test_query_texts,
    batch_size=128,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

# 3. Retrieval.
max_k = 5
requests = [
    models.QueryRequest(
        query=embedding.tolist(),
        limit=max_k,
        with_payload=False,
    )
    for embedding in query_embeddings
]
responses = client.query_batch_points(
    collection_name=collection_name,
    requests=requests,
)

# 4. Ground truth -> ir_measures.
eval_qrels = [
    ir_measures.Qrel(
        query_id=str(row['query-id']),
        doc_id=str(row['corpus-id']),
        relevance=int(row['score']),
    )
    for row in qrels
]

# 5. Response -> ir_measures.
run = []
for query_id, response in zip(test_query_ids, responses):
    for point in response.points:
        run.append(
            ir_measures.ScoredDoc(
                query_id=query_id,
                doc_id=str(point.id),
                score=float(point.score),
            )
        )

# 6. Evaluate.
metrics = ir_measures.calc_aggregate(
    [
        Success@5,      # hit rate@5.
        R@5,            # recall@5.
        R@10,           # recall@10.
        RR@10,          # mrr@10.
        nDCG@10,        # nDCG.
    ],
    eval_qrels,
    run,
)

for metric, value in metrics.items():
    print(f'{metric}: {value:.4f}')

Test queries: 300


Batches: 100%|██████████| 3/3 [00:00<00:00, 30.36it/s]


Success@5: 0.7833
R@5: 0.7653
nDCG@10: 0.6922
RR@10: 0.6743
R@10: 0.7653
